# Stage 3 · Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedzait20003/TableMind/blob/main/src/notebooks/03_evaluation.ipynb)

Loads a pretrained run from Drive and evaluates it on the full WikiSQL test split: loss and token accuracy, then **denotation accuracy** (the metric DoT reports: predicted answer values must equal the executed SQL answer as a set, numbers compared numerically), exact match, a per-aggregation breakdown, and a comparison across runs with the DoT paper.

**Requires:** `02_pretrain.ipynb`

## Setup

On Colab, pick **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
# Environment setup: clones the repo on Colab, uses the local checkout otherwise
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/mohamedzait20003/TableMind.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/TableMind"):
        !git clone -q {REPO_URL} /content/TableMind
    %cd /content/TableMind
    !pip install -q -r requirements.txt
else:
    # Local run (e.g. from src/notebooks): move up to the repository root
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()))

sys.path.insert(0, os.path.abspath("src"))
print("Working directory:", os.getcwd())

In [ ]:
from utils import load_config, set_seed, resolve_device, get_storage_dir

# DoT setup: tapas-small prunes 1024 -> 256 tokens for a t5-base task model,
# trained on the full WikiSQL training set
CONFIG_PATH = "src/config/default.yaml"

config = load_config(CONFIG_PATH)
device = resolve_device(config["system"]["device"])
set_seed(config["system"]["seed"])

# Google Drive on Colab (you will be asked to authorize), ./storage locally
STORAGE = get_storage_dir(config["storage"]["drive_subdir"], config["storage"]["local_dir"])
CHECKPOINT_DIR = config["system"]["checkpoint_dir"]
OUTPUT_DIR = config["system"]["output_dir"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config: {CONFIG_PATH} ({config['experiment']['name']})")
print(f"Device: {device} | storage: {STORAGE}")

## Run settings

Pick the run to evaluate (same values as in `02_pretrain.ipynb`).

In [ ]:
from utils import run_name

# DoT reports the median of 3 runs: train and evaluate with SEED = 42, 43, 44.
# USE_MEMORY = False trains the ablation without memory retrieval (all else equal).
# Use the same two values in 02_pretrain and 03_evaluation.
SEED = 42
USE_MEMORY = True

config["system"]["seed"] = SEED
config["model"]["task_transformer"]["use_memory"] = USE_MEMORY
set_seed(SEED)
RUN = run_name(config)
print("Run:", RUN)

## Load the pretrained model

In [ ]:
from core import MemoryEncoder, KeyValueMemoryStore, DoTModel
from utils import download_checkpoint, load_checkpoint

memory_encoder = MemoryEncoder.from_config(config)
memory_store = KeyValueMemoryStore()
model = DoTModel.from_config(config, memory_store, memory_encoder)

model_path = download_checkpoint(f"{RUN}_final.pt", STORAGE, CHECKPOINT_DIR)
load_checkpoint(model_path, model, memory_store=memory_store)
model.to(device).eval()
print("Memory entries:", memory_store.size(), "| retrieval enabled:", USE_MEMORY)

## Load the WikiSQL test split

In [ ]:
from torch.utils.data import DataLoader
from utils import WikiSQLDataset, collate_fn

test_dataset = WikiSQLDataset.from_config(config, "test")
test_loader = DataLoader(test_dataset, batch_size=config["pretraining"]["batch_size"],
                         collate_fn=collate_fn, num_workers=config["system"]["num_workers"])
print("Test examples:", len(test_dataset))

## Loss and token accuracy

Teacher-forced: each answer token is predicted from the gold previous tokens.

In [ ]:
from core import PretrainTrainer

evaluator = PretrainTrainer(model, None, test_loader, device=device)
loss_metrics = evaluator.evaluate()
for name, value in loss_metrics.items():
    print(f"{name}: {value:.4f}")

## How much does the model use memory?

The table embedding and the retrieved memory are added to T5's encoder states through gates that start at 0 (`tanh(0) = 0`, so training begins from plain T5). The learned values show how much of each signal the model chose to mix in; `memory_gate` stays at 0 in the no-memory ablation.

In [ ]:
gates = model.task_model.gate_values()
for name, value in gates.items():
    print(f"{name}: {value:+.4f}")

## Denotation accuracy and exact match

Generated answers are split on ` | ` and compared with the executed SQL answer as a multiset; numbers are compared numerically ("1,234" = "1234.0"). Set `MAX_BATCHES` to an integer for a quick partial check.

In [ ]:
import pandas as pd
from utils import evaluate_answers

MAX_BATCHES = None

answer_metrics, records = evaluate_answers(
    model, test_loader, device,
    max_new_tokens=config["data"]["wikisql"]["answer_max_length"],
    max_batches=MAX_BATCHES,
)
print(f"Denotation accuracy: {answer_metrics['denotation_accuracy']:.4f}")
print(f"Exact match:         {answer_metrics['exact_match']:.4f}")
print(f"Examples:            {answer_metrics['examples']}")

results = pd.DataFrame(records)
# test_loader is not shuffled, so record i is test example i
results["question"] = [test_dataset.data[i].question for i in range(len(results))]
results["aggregation"] = [test_dataset.data[i].aggregation.name for i in range(len(results))]
results[["question", "answer", "prediction", "denotation"]].head(20)

In [ ]:
results.groupby("aggregation")["denotation"].agg(denotation_accuracy="mean", examples="count")

## Save this run's results

Predictions and metrics are uploaded to Drive so runs can be compared below.

In [ ]:
import json
from utils import upload_checkpoint

predictions_path = os.path.join(OUTPUT_DIR, f"{RUN}_predictions.csv")
results.to_csv(predictions_path, index=False)
upload_checkpoint(predictions_path, STORAGE)

summary = {"run": RUN, "seed": SEED, "use_memory": USE_MEMORY, **loss_metrics, **answer_metrics, **gates}
summary_path = os.path.join(OUTPUT_DIR, f"{RUN}_results.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
upload_checkpoint(summary_path, STORAGE)
summary

## Compare runs with the DoT paper

Collects every `*_results.json` in Drive. DoT reports the **median of 3 runs**, so evaluate seeds 42, 43 and 44 (with and without memory) before drawing conclusions. Paper numbers: WikiSQL test denotation accuracy, Table 1 of Krichene et al. (2021).

In [ ]:
import glob

runs = pd.DataFrame([json.load(open(p)) for p in glob.glob(os.path.join(STORAGE, "*_results.json"))])
ours = (runs.groupby("use_memory")["denotation_accuracy"]
            .agg(median="median", min="min", max="max", runs="count") * [100, 100, 100, 1])
ours.index = [f"TableMind (T5{' + memory' if m else ''})" for m in ours.index]

paper = pd.DataFrame(
    {"median": [83.6, 86.0, 85.3, 85.5], "runs": [3, 3, 3, 3]},
    index=["TAPAS large, 512 tokens (paper)", "TAPAS large, 1024 tokens (paper)",
           "DoT small→256→large, HEM (paper)", "DoT medium→256→large, HEM (paper)"],
)
display(runs[["run", "denotation_accuracy", "exact_match", "eval_loss", "memory_gate"]].sort_values("run"))
pd.concat([ours, paper]).round(2)

## Inspect pruning on one example

What T5 sees after TAPAS keeps whole cells, with each body cell's `sigmoid(score)` scale (question and header are kept unscaled).

In [ ]:
import torch

batch = next(iter(test_loader))
with torch.no_grad():
    t5_ids, t5_mask, scale = model.task_inputs(
        batch["input_ids"][:1].to(device),
        batch["attention_mask"][:1].to(device),
        batch["token_type_ids"][:1].to(device),
    )

tokenizer = model.task_model.tokenizer
print("Question:", test_dataset.data[0].question)
print("Answer:  ", test_dataset.data[0].answer)
print("\nT5 input:", tokenizer.decode(t5_ids[0], skip_special_tokens=True))

tokens = tokenizer.convert_ids_to_tokens(t5_ids[0][t5_mask[0].bool()].tolist())
weights = scale[0][t5_mask[0].bool()].tolist()
pd.DataFrame({"token": tokens, "scale": weights}).query("scale < 1").sort_values("scale", ascending=False).head(15)